<a href="https://colab.research.google.com/github/aiyman14/DACSS-758-Text-as-Data/blob/main/Final_project_check-in_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project Check In #2
## DACSS 758: Text as Data
## Aiyman Akbar

## Check In #1 Summary

The corpus that I am using for this project consists of news articles related to Bitcoin and cryptocurrency. I got this dataset from Hugging Face and it contains approximately 2,500 news articles. The articles are collected from various online news sources, including financial news websites, cryptocurrency publications, and general news outlets. Each entry in the dataset is a collection of different things from the article itself, such as the article title, summary text and author. These articles appear to have been aggregated from online news reporting about Bitcoin and cryptocurrency markets.

This dataset was probably created to support research in NLP, financial texts and analyzing them and related fields of research. Because it is an aggregation of crypto related articles it is easy to understand the relationship between digital assets and the media and how news narratives may influence market perception. However, the dataset may contain several sources of bias. Because news coverage tends to focus on major events such as market crashes or technological breakthroughs, this may lead to an overrepresentation of dramatic or high-impact stories. Additionally, different news outlets frame cryptocurrency differently depending on their audience or editorial stance.

I expect the language in the corpus to be pretty formal, in a journalistic style about financial and tech news. These types of articles generally do include jargon and specific terminology related to cryptocurrency, the markets, prices etc. I find this corpus interesting because of the relationship that it allows you to see between cryptocurrency and the media, and specifically how news narratives can influence the price of crypto and vice versa how the price fluctuations of crypto change the public sentiment or news headlines related to cryptocurrency. The corpus is structured as a standard dataset with each row being a single news article with many fields as the columns, and the total word count is around 230,813 words.

## Setup and Loading Data

In [ ]:
# install required libraries
!pip install datasets gensim wordcloud

In [ ]:
# imports
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from wordcloud import WordCloud

# for preprocessing
from gensim.parsing.preprocessing import preprocess_string, strip_tags, strip_punctuation, strip_short
from gensim.parsing.preprocessing import strip_multiple_whitespaces, strip_numeric, remove_stopwords

In [ ]:
# load dataset directly from Hugging Face
ds = load_dataset("khaihernlow/bitcoin-news-articles-text-corpora")
df = pd.DataFrame(ds["train"])

In [ ]:
# explore the dataset
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head()

In [ ]:
# check for null values in key text columns
print("Null values in summary:", df['summary'].isnull().sum())
print("Null values in title:", df['title'].isnull().sum())
print("\nTopic distribution:")
print(df['topic'].value_counts())

In [ ]:
# drop rows with missing summaries since that is our main text field
df = df.dropna(subset=['summary'])
print("Shape after dropping nulls:", df.shape)

## Preprocessing

For this dataset, I am using the article `summary` field as my main text for analysis since it contains the most substantial text content for each article. The unit of analysis is a single article, represented by its summary. Below I will walk through each preprocessing step I chose to complete and explain why.

First, here is an example of what one article summary looks like before any preprocessing:

In [ ]:
# example article BEFORE preprocessing
summaries = df['summary'].tolist()
example_idx = 0

print("=" * 60)
print("BEFORE PREPROCESSING")
print("=" * 60)
print(summaries[example_idx])

# Step 1: lowercase
summaries_lower = [item.lower() for item in summaries]

In [ ]:
# Step 1: lowercase
summaries = df['summary'].tolist()
summaries_lower = [item.lower() for item in summaries]

### Step 2: Remove Punctuation

Punctuation does not really add any meaning for the kind of text analysis I am doing here. Things like commas, periods, quotation marks etc. would just add noise to the data and mess up tokenization, so I am stripping all of that out.

### Step 3: Remove Extra Whitespace

After removing punctuation and other characters there can be extra spaces left over. Stripping multiple whitespaces just cleans things up and makes sure tokens are separated properly.

### Step 4: Remove Stopwords

Stopwords are common words like "the", "is", "and", "of" etc. that appear very frequently but do not carry much meaning for analysis. If I kept them in they would dominate the word frequencies and make it harder to see the actual important terms related to Bitcoin and cryptocurrency news. Removing them lets the more meaningful content words stand out.

### Step 5: Remove Short Words

I am also removing words that are shorter than 3 characters. These are usually leftover fragments or very common words that do not add value to the analysis. This is a step that was done in the class labs and it helps to clean up the data even more.

In [ ]:
# Steps 2-5: apply preprocessing filters
CUSTOM_FILTERS = [strip_punctuation, strip_multiple_whitespaces, remove_stopwords, strip_short]
summaries_toks = [preprocess_string(item, CUSTOM_FILTERS) for item in summaries_lower]
print(summaries_toks[0])

### Deliberately Omitted Steps

I chose not to do stemming or lemmatization at this stage. Stemming can sometimes cut words in ways that make them harder to read and interpret, and for my purposes right now I want the words to still be recognizable when I look at frequency plots and word clouds. For example, stemming might turn "cryptocurrency" into "cryptocurrenc" which is not as easy to understand at a glance. I also did not remove numbers because in financial news articles numbers like prices and percentages can be meaningful, but I may reconsider this depending on how the analysis goes.

In [ ]:
## After Preprocessing Example

Now here is the same article after all the preprocessing steps have been applied:

# same article AFTER preprocessing
print("=" * 60)
print("AFTER PREPROCESSING")
print("=" * 60)
print(summaries_toks_merged[example_idx])

In [ ]:
# pick an example article
example_idx = 0

print("=" * 60)
print("BEFORE PREPROCESSING")
print("=" * 60)
print(summaries[example_idx])

print()

print("=" * 60)
print("AFTER PREPROCESSING")
print("=" * 60)
print(summaries_toks_merged[example_idx])

## Visualizations

### Visualization 1: Top 25 Most Frequent Words

This bar chart shows the 25 most common words across all the article summaries after preprocessing. This is helpful because it gives a clear picture of what the dominant themes and topics are in Bitcoin news coverage. If certain words show up way more than others, that tells us something about what the media focuses on when they talk about Bitcoin and cryptocurrency.

In [ ]:
# unnest all tokens into one list
all_tokens = [item for sublist in summaries_toks for item in sublist]
print("Total tokens:", len(all_tokens))

# count frequencies
c = Counter(all_tokens)
word_df = pd.DataFrame(c.items(), columns=['Token', 'Count'])
word_df = word_df.sort_values(by='Count', ascending=False)
word_df.head(10)

In [ ]:
# plot top 25 most frequent words
top25 = word_df.head(25)

plt.figure(figsize=(12, 6))
plt.bar(top25['Token'], top25['Count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Token')
plt.ylabel('Frequency')
plt.title('Top 25 Most Frequent Words in Bitcoin News Summaries')
plt.tight_layout()
plt.show()

### Visualization 2: Word Cloud

The word cloud is a good way to visualize the same frequency data but in a more visually interesting way. The size of each word corresponds to how often it appears in the corpus, so the biggest words are the most common ones. This makes it easy to see at a glance what the most prominent terms are in the dataset. It is also just a nice visual to include in a report because it immediately communicates what the corpus is about.

In [ ]:
# create word cloud from all preprocessed text
all_text = ' '.join(summaries_toks_merged)

wordcloud = WordCloud(width=800, height=400,
                      background_color='white',
                      colormap='viridis',
                      max_words=100).generate(all_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Bitcoin News Article Summaries')
plt.tight_layout()
plt.show()

## Next Steps

Moving forward with this data, I would like to explore sentiment analysis of the Bitcoin news articles. One potential approach would be to use a sentiment dictionary like VADER or AFINN (which we used in the class labs) to score each article summary and see the overall distribution of positive, negative and neutral sentiment across the corpus. This would help answer the question of whether Bitcoin news coverage tends to lean more positive or negative overall.

Another thing I am interested in is looking at how the language and sentiment might differ across the different topic categories in the dataset, like "news" vs "finance" vs "tech". Since different types of outlets and topics might frame Bitcoin differently, comparing the sentiment and word usage across these categories could reveal some interesting patterns about how Bitcoin is discussed in different contexts.

I am also thinking about looking at word embeddings using something like Doc2Vec (which we also covered in class) to see how words relate to each other in the context of Bitcoin news. This could help identify clusters of related terms and themes within the corpus and give a deeper understanding of the narrative structure of cryptocurrency news coverage.

## Challenges

One challenge I encountered while working with this data is dealing with missing values. There are 27 articles that have null summaries, which I had to drop from the dataset. While this is a small number out of 2,500 it is still something to keep in mind because those missing articles might represent a certain type of coverage that is now not included in the analysis. I also noticed that some other columns like author and excerpt have missing values too which could be a limitation if I want to do any analysis based on those fields.

Another challenge is the potential for domain-specific terms to affect the analysis. Words like "bitcoin", "crypto", and "blockchain" are going to dominate the frequency counts because that is literally what every article is about, so it might be worth considering whether to filter those out in future analysis to see what other themes emerge beyond the obvious ones. Additionally, the articles are from specific news sources which means there could be some repetition or overlap in coverage of the same events, which might skew the results.

Going forward, I anticipate that choosing the right approach for sentiment analysis might be tricky. General purpose sentiment dictionaries might not capture the nuances of financial and crypto-specific language very well. For example, a word like "crash" has a very negative connotation in general but in crypto news it is used pretty matter-of-factly to describe price movements. This is something I will need to think about as I move into the analysis phase of the project.